# Online Retail II — reproducible transaction analysis

Source: Daqing Chen, UCI, DOI 10.24432/C5CG6D, CC BY 4.0. Run `python scripts/download_data.py` first. Amounts are GBP; cancellations are distinct C-prefixed invoices. Net revenue includes signed returns. This is descriptive analysis, not causal business impact.

In [1]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
from analytics import load, prepare, monthly_net, cancellation_counts, customer_rfm
path = Path(os.getenv("RETAIL_INPUT", "data/raw/online_retail_II.xlsx"))
df = prepare(load(path))
print("Rows after exact deduplication:", len(df))
print("Net revenue GBP:", round(df["Total_Price"].sum(), 2))

Rows after exact deduplication: 1033036
Net revenue GBP: 18855533.7


## Monthly net revenue
Sum transaction values before plotting; do not plot their default mean. Partial first/last months should not be compared as full months.

In [2]:
monthly = monthly_net(df)
monthly.plot(title="Monthly net revenue (GBP)", ylabel="GBP", figsize=(12,4))
plt.tight_layout(); plt.show()
print(monthly.to_string())

InvoiceDate
2009-12     796648.500
2010-01     622479.502
2010-02     531265.366
2010-03     763247.241
2010-04     587926.692
2010-05     613270.720
2010-06     677073.870
2010-07     573333.690
2010-08     654774.390
2010-09     851105.961
2010-10    1041685.610
2010-11    1416697.202
2010-12     746723.610
2011-01     558448.560
2011-02     497026.410
2011-03     682013.980
2011-04     492367.841
2011-05     722094.100
2011-06     689977.230
2011-07     680156.991
2011-08     681386.460
2011-09    1017596.682
2011-10    1069368.230
2011-11    1456145.800
2011-12     432719.060
Freq: M


In [3]:
print("Distinct cancelled invoices by year:")
print(cancellation_counts(df))
print("All distinct invoices:", df["Invoice"].nunique())

Distinct cancelled invoices by year:
Year
2009     401
2010    4381
2011    3510
Name: Invoice, dtype: int64
All distinct invoices: 53628


## Customers and products
Revenue totals retain anonymous customers. RFM uses known customers and positive non-cancelled purchases only; monetary value is gross purchase value, not net value after returns.

In [4]:
rfm = customer_rfm(df)
print(rfm.describe())
df.groupby("Description")["Total_Price"].sum().nlargest(10).sort_values().plot.barh(title="Top products by net revenue (GBP)")
plt.tight_layout(); plt.show()

         frequency       monetary  recency_days
count  5878.000000    5878.000000   5878.000000
mean      6.289384    2955.904095    200.866791
std      13.009406   14440.852688    209.353961
min       1.000000       2.950000      0.000000
25%       1.000000     342.280000     25.000000
50%       3.000000     867.740000     95.000000
75%       7.000000    2248.305000    379.000000
max     398.000000  580987.040000    738.000000


In [5]:
Path("reports").mkdir(exist_ok=True)
df.to_csv("reports/cleaned_transactions.csv", index=False)
monthly.to_csv("reports/monthly_net_gbp.csv")
print("Wrote local analysis outputs; raw workbook unchanged.")

Wrote local analysis outputs; raw workbook unchanged.


## SQL and dashboard assets
Import `reports/cleaned_transactions.csv` into SQL Server as `sales_table` before using the analysis script. Existing screenshots are retained; no editable Power BI source is currently supplied. Customer-level exports stay local and are ignored.